In [1]:
from trustifai import Trustifai, MetricContext

# Confidence for LLM response

_works for only those LLMs which supports logprobs_

In [11]:
trust_engine = Trustifai("config_file.yaml")

In [3]:
response = trust_engine.generate("what was the major means of transport in the year 2150? Why it got failed?")

In [11]:
response

{'response': 'As of now (2024), the year 2150 is in the future, and there is no historical record or factual information about the major means of transport in that year. Any discussion about transportation in 2150 would be speculative or based on science fiction.\n\nIf you are referring to a specific book, movie, or fictional universe, please clarify so I can provide an answer based on that context. Otherwise, in real-world history, we cannot say what the major means of transport in 2150 was or why it might have failed.',
 'metadata': {'confidence_score': 0.71,
  'confidence_label': 'Medium Confidence',
  'confidence_details': {'explanation': 'Model shows moderate uncertainty.',
   'avg_logprob': -0.24,
   'variance': 0.09,
   'token_count': 109},
  'logprobs_available': True,
  'execution_metadata': {'total_cost_usd': '0.000952'}}}

# Trust Score for RAG response

In [2]:
from langchain_core.documents import Document

query = "What is Acme Corp's policy on remote work?"

answer = """Acme Corp operates on a fully remote-first policy with no office requirements. 
Employees can work from anywhere in the world and the company has closed all 
physical office locations as of January 2024. They provide a $2000 annual 
stipend for coworking spaces and have implemented a 4-day work week.
"""


documents = [
    Document(
        page_content="Acme Corp announced a hybrid work model in 2023, requiring employees to be in office 3 days per week.",
        metadata={"source": "hr_policy_2023.pdf"}
    ),
    Document(
        page_content="The company provides home office stipends of up to $500 for remote setup.",
        metadata={"source": "benefits_guide.pdf"}
    )
]
# documents = ["Acme Corp announced a hybrid work model in 2023, requiring employees to be in office 3 days per week.",
#              "The company provides home office stipends of up to $500 for remote setup."]

#supports document objects like langchain/llamaindex documents, list, dictionary etc.


trust_engine = Trustifai("config_file.yaml")

metric_context = MetricContext(
    query=query,
    answer=answer,
    documents=documents,
)

In [3]:
trust_score = trust_engine.get_trust_score(metric_context)

In [6]:
trust_score

{'score': 0.44,
 'label': 'UNRELIABLE',
 'details': {'evidence_coverage': {'score': 0.0,
   'label': 'Likely Hallucinated Answer',
   'details': {'explanation': 'Many claims lack support from source documents.',
    'strategy': 'LLM',
    'total_sentences': 3,
    'supported_sentences': 0,
    'unsupported_sentences': ['Acme Corp operates on a fully remote-first policy with no office requirements.',
     'Employees can work from anywhere in the world and the company has closed all \nphysical office locations as of January 2024.',
     'They provide a $2000 annual \nstipend for coworking spaces and have implemented a 4-day work week.'],
    'failed_checks': 0},
   'execution_metadata': {'total_cost_usd': 0.002542}},
  'semantic_drift': {'score': 0.72,
   'label': 'Partial Alignment',
   'details': {'explanation': 'Some claims may not be aligned with source documents.',
    'total_documents': 2,
    'total_sentences_checked': 2,
    'best_matching_sentence': 'Acme Corp announced a hybrid

In [6]:
rg = trust_engine.build_reasoning_graph(trust_score)

In [7]:
trust_engine.visualize(rg)

<class 'pyvis.network.Network'> |N|=6 |E|=5

In [8]:
print(trust_engine.visualize(rg,"mermaid"))

```mermaid
flowchart TD
   evidence_coverage["<b>Evidence Coverage</b><br/>Score: 0.00<br/>Likely Hallucinated Answer"]
   semantic_drift["<b>Semantic Drift</b><br/>Score: 0.72<br/>Partial Alignment"]
   consistency["<b>Consistency</b><br/>Score: 0.62<br/>Fragile Consistency"]
   source_diversity["<b>Source Diversity</b><br/>Score: 0.85<br/>High Trust"]
   trust_aggregation{"<b>Trust Score</b><br/>Score: 0.42"}
   final_decision("<b>Decision: UNRELIABLE</b>")
    evidence_coverage --> trust_aggregation
    semantic_drift --> trust_aggregation
    consistency --> trust_aggregation
    source_diversity --> trust_aggregation
    trust_aggregation --> final_decision
    style evidence_coverage fill:#ff6b6b,color:#000000
    style semantic_drift fill:#f39c12,color:#000000
    style consistency fill:#f39c12,color:#000000
    style source_diversity fill:#2ecc71,color:#000000
    style trust_aggregation fill:#ff6b6b,color:#000000
    style final_decision fill:#ff6b6b,color:#000000
```
